In [1]:
import json
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT_DIR = Path("/home/yangdejin/nlpcc/nlpcc_task2")
DATA_DIR = ROOT_DIR / "data"
OUTPUTS_DIR = ROOT_DIR / "outputs"
DEV_FILE = DATA_DIR / "dev.jsonl"


TRACK2_MODEL_WAY = 'grpo'
TRACK2_MODEL_NAME = "qwen35_9b"
TRACK2_MODEL_DIR = OUTPUTS_DIR / "track2" / TRACK2_MODEL_NAME / TRACK2_MODEL_WAY
EVAL_OUT_DIR = OUTPUTS_DIR / 'track2' / TRACK2_MODEL_NAME /'eval_based_moco_scl'/ TRACK2_MODEL_WAY
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
TRACK2_BASE_MODEL = "Qwen/Qwen3.5-9B"

TRACK1_MODEL_DIR = OUTPUTS_DIR / 'track1' / 'encoder' / 'deberta_v2_xxlarge_moco_scl' / 'best_macro_f1_model'
TRACK1_BASE_MODEL = "microsoft/deberta-v2-xxlarge"

MAX_PROMPT_LENGTH = 640
MAX_NEW_TOKENS = 96
TRACK1_MAX_LENGTH = 512
TRACK1_BATCH_SIZE = 32

NUM_CANDIDATES = 4
TEMPERATURE = 0.7
TOP_P = 0.9

EXPERIMENT_NAME = f"{TRACK2_MODEL_WAY}_sample{NUM_CANDIDATES}"
USE_CACHE = True
GENERATION_FILE = EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_dev_generations.jsonl'
DETAIL_FILE = EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_track1_eval_details.csv'
print('DEV_FILE:', DEV_FILE)
print('TRACK2_MODEL_DIR:', TRACK2_MODEL_DIR)
print('TRACK1_MODEL_DIR:', TRACK1_MODEL_DIR)
print('GENERATION_FILE:', GENERATION_FILE)

DEV_FILE: /home/yangdejin/nlpcc/nlpcc_task2/data/dev.jsonl
TRACK2_MODEL_DIR: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/grpo
TRACK1_MODEL_DIR: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track1/encoder/deberta_v2_xxlarge_moco_scl/best_macro_f1_model
GENERATION_FILE: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/eval_based_moco_scl/grpo/qwen35_9b_dev_generations.jsonl


In [2]:
# Cell 2: Official values, IO, prompt builder

VALUE_DEFINITIONS = {
    'Self-direction–thought': "Freedom to cultivate one's own ideas and abilities.",
    'Self-direction–action': "Freedom to determine one's own actions.",
    'Stimulation': 'Excitement, novelty, and change.',
    'Hedonism': 'Pleasure and sensuous gratification.',
    'Achievement': 'Success according to social standards.',
    'Power–dominance': 'Power through exercising control over people.',
    'Power–resources': 'Power through control of material and social resources.',
    'Face': "Security and power through maintaining one's public image and avoiding humiliation.",
    'Security–personal': "Safety in one's immediate environment.",
    'Security–societal': 'Safety and stability in the wider society.',
    'Tradition': 'Maintaining and preserving cultural, family, or religious traditions.',
    'Conformity–rules': 'Compliance with rules, laws, and formal obligations.',
    'Conformity–interpersonal': 'Avoidance of upsetting or harming other people.',
    'Humility': "Recognizing one's insignificance in the larger scheme of things.",
    'Benevolence–dependability': 'Being a reliable and trustworthy member of the ingroup.',
    'Benevolence–caring': 'Devotion to the welfare of ingroup members.',
    'Universalism–concern': 'Commitment to equality, justice, and protection for all people.',
    'Universalism–nature': 'Preservation of the natural environment.',
    'Universalism–tolerance': 'Acceptance and understanding of those who are different from oneself.',
}
VALUE_LABELS = list(VALUE_DEFINITIONS.keys())
label2id = {v: i for i, v in enumerate(VALUE_LABELS)}
id2label = {i: v for i, v in enumerate(VALUE_LABELS)}

TASK_INSTRUCTION = (
    'You are given a scenario, a question, and a target human value. '
    'Generate one concise, meaningful response that answers the question, '
    'fits the scenario, and naturally aligns with the target value.'
)
def normalize_space(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def build_prompt(row, use_value_definition=True):
    scenario = normalize_space(row.get('Scenario', ''))
    question = normalize_space(row.get('Question', ''))
    value = normalize_space(row.get('Value', ''))
    parts = [
        TASK_INSTRUCTION,
        f'Scenario:\n{scenario}',
        f'Question:\n{question}',
        f'Target value:\n{value}',
    ]
    if use_value_definition:
        parts.append(f'Target value definition:\n{VALUE_DEFINITIONS.get(value, "")}')
    parts.append('Response:')
    return '\n\n'.join(parts) + '\n'

def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
dev_rows = read_jsonl(DEV_FILE)
for i, row in enumerate(dev_rows):
  row['idx'] = i
  row["prompt"] = build_prompt(row)
print("dev rows: ", len(dev_rows))
print('sample prompt:\n', dev_rows[0]['prompt'][:700])


dev rows:  514
sample prompt:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
A teammate insists on frequent communication to feel secure.

Question:
How would you handle the teammate’s demand for constant updates?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:



In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_track2_model(model_dir):
    model_dir = Path(model_dir)
    tokenizer_src = str(model_dir) if model_dir.exists() else TRACK2_BASE_MODEL

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_src, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    model_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtypr = "bf16",
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = "auto"

    if (model_dir / "adapter_config.json").exists():
        base_model = AutoModelForCausalLM.from_pretrained(TRACK2_BASE_MODEL, **model_kwargs)
        base_model.resize_token_embeddings(len(tokenizer))
        model = PeftModel.from_pretrained(base_model, str(model_dir))
    else:
        model = AutoModelForCausalLM.from_pretrained(TRACK2_BASE_MODEL, **model_kwargs)
        model.resize_token_embeddings(len(tokenizer))

    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model, tokenizer


def is_valid_candidate(text, value):
    text = normalize_space(text)
    if not text:
        return False
    words = text.split()
    if len(words) < 8 or len(words) > 80:
        return False
    bad_phrases = ["Scenario:", "Question:", "Target value:", "Response:"]
    if any(x in text for x in bad_phrases):
        return False
    if value in text:
        return False
    return True


def generate_candidates(model, tokenizer, prompt, value, n=4):
    device = next(model.parameters()).device
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            num_return_sequences=n,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    candidates = []
    for ids in outputs[:, prompt_len:]:
        text = normalize_space(tokenizer.decode(ids, skip_special_tokens=True))
        if is_valid_candidate(text, value):
            candidates.append(text)

    candidates = list(dict.fromkeys(candidates))
    if not candidates:
        candidates = ["I would respond in a way that answers the question while respecting the target value."]
    return candidates


if USE_CACHE and GENERATION_FILE.exists():
    generated_rows = read_jsonl(GENERATION_FILE)
    print("Loaded cached candidates:", GENERATION_FILE)
else:
    model, tokenizer = load_track2_model(TRACK2_MODEL_DIR)
    generated_rows = []

    for row in tqdm(dev_rows):
        candidates = generate_candidates(
            model=model,
            tokenizer=tokenizer,
            prompt=row["prompt"],
            value=row["Value"],
            n=NUM_CANDIDATES,
        )
        generated_rows.append({
            "idx": row["idx"],
            "Value": row["Value"],
            "candidates": candidates,
            "num_candidates": len(candidates),
        })

    write_jsonl(GENERATION_FILE, generated_rows)
    print("Saved candidates:", GENERATION_FILE)

print("generated size:", len(generated_rows))
print(generated_rows[0])

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

  0%|          | 0/514 [00:00<?, ?it/s]

Saved candidates: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/eval_based_moco_scl/grpo/qwen35_9b_dev_generations.jsonl
generated size: 514
{'idx': 0, 'Value': 'Conformity–interpersonal', 'candidates': ['I would agree to the frequent updates to ensure the teammate feels safe and respected, prioritizing harmony and avoiding any conflict that might upset them.', 'I would prioritize maintaining harmony by accommodating their need for frequent updates, even if it means adjusting my own schedule, to avoid causing them discomfort or conflict.', 'I would accommodate their need for frequent communication to maintain harmony in the relationship, prioritizing their emotional comfort and avoiding any conflict that might hurt their feelings.', 'I would agree to provide regular updates and avoid being dismissive of their need for reassurance, as I want to maintain harmony in the relationship and prevent any feelings of being disregarded.'], 'num_candidates': 4}


In [4]:
import torch
import torch.nn as nn
from types import SimpleNamespace
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class Track1CompactClassifier(nn.Module):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        pooled = mean_pooling(outputs.last_hidden_state, attention_mask)
        pooled = pooled.to(self.classifier.weight.dtype)
        logits = self.classifier(pooled)
        return SimpleNamespace(logits=logits)


def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_track1_model(model_dir):
    model_dir = Path(model_dir)
    if not model_dir.exists() and (model_dir.parent / "best_macro_f1_model").exists():
        model_dir = model_dir.parent / "best_macro_f1_model"
    if not model_dir.exists():
        raise FileNotFoundError(
            f"Track1 best model directory not found: {model_dir}. "
            "Run the modified MoCo-SCL training notebook first."
        )

    heads_path = model_dir / "heads.pt"
    adapter_dir = model_dir / "query_encoder_lora"
    if not heads_path.exists() or not adapter_dir.exists():
        raise FileNotFoundError(
            f"Expected compact Track1 files under {model_dir}: heads.pt and query_encoder_lora/."
        )

    heads = torch_load_compat(heads_path, map_location="cpu")
    base_model_name = heads.get("model_name", TRACK1_BASE_MODEL)
    num_labels = int(heads.get("num_labels", len(VALUE_LABELS)))

    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)
    model_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    encoder = AutoModel.from_pretrained(base_model_name, **model_kwargs)
    encoder = PeftModel.from_pretrained(encoder, str(adapter_dir))

    hidden_size = getattr(getattr(encoder, "config", None), "hidden_size", None)
    if hidden_size is None:
        hidden_size = encoder.base_model.model.config.hidden_size
    classifier = nn.Linear(hidden_size, num_labels)
    classifier.load_state_dict(heads["classifier"])

    model = Track1CompactClassifier(encoder, classifier)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    metric_file = model_dir / "best_metric.json"
    if metric_file.exists():
        with open(metric_file, encoding="utf-8") as f:
            print("Track1 best metric:", json.load(f))
    print("Loaded Track1 compact model from:", model_dir)
    return model, tokenizer, device


track1_model, track1_tokenizer, track1_device = load_track1_model(TRACK1_MODEL_DIR)


def classify_batch(texts):
    inputs = track1_tokenizer(
        ["Response: " + normalize_space(t) for t in texts],
        padding=True,
        truncation=True,
        max_length=TRACK1_MAX_LENGTH,
        return_tensors="pt",
    )
    inputs = {k: v.to(track1_device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = track1_model(**inputs).logits.float()
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    return probs


flat_items = []
for row in generated_rows:
    for cand_id, candidate in enumerate(row["candidates"]):
        flat_items.append({
            "idx": row["idx"],
            "Value": row["Value"],
            "cand_id": cand_id,
            "candidate": candidate,
        })

all_probs = []
texts = [x["candidate"] for x in flat_items]

for start in tqdm(range(0, len(texts), TRACK1_BATCH_SIZE)):
    batch_probs = classify_batch(texts[start:start + TRACK1_BATCH_SIZE])
    all_probs.append(batch_probs)

probs = np.concatenate(all_probs, axis=0)

candidate_records = []
for item, prob in zip(flat_items, probs):
    target_id = label2id[item["Value"]]
    pred_id = int(prob.argmax())

    sorted_ids = np.argsort(prob)[::-1]
    best_other_id = next(i for i in sorted_ids if i != target_id)

    target_prob = float(prob[target_id])
    pred_prob = float(prob[pred_id])
    margin = float(prob[target_id] - prob[best_other_id])

    words = len(item["candidate"].split())
    length_penalty = 0.0
    if words < 15:
        length_penalty = 0.05
    elif words > 55:
        length_penalty = 0.03

    rerank_score = target_prob + 0.25 * margin - length_penalty

    candidate_records.append({
        "idx": item["idx"],
        "Value": item["Value"],
        "cand_id": item["cand_id"],
        "candidate": item["candidate"],
        "pred_value": id2label[pred_id],
        "is_match": id2label[pred_id] == item["Value"],
        "target_prob": target_prob,
        "pred_prob": pred_prob,
        "margin": margin,
        "word_count": words,
        "rerank_score": rerank_score,
    })

candidate_df = pd.DataFrame(candidate_records)
display(candidate_df.head())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/778 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v2-xxlarge
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Track1 best metric: {'best_macro_f1': 0.9417667224757209, 'global_step': 770, 'epoch': 14.0, 'model_name': 'microsoft/deberta-v2-xxlarge', 'num_labels': 19, 'labels': ['Self-direction–thought', 'Self-direction–action', 'Stimulation', 'Hedonism', 'Achievement', 'Power–dominance', 'Power–resources', 'Face', 'Security–personal', 'Security–societal', 'Tradition', 'Conformity–rules', 'Conformity–interpersonal', 'Humility', 'Benevolence–dependability', 'Benevolence–caring', 'Universalism–concern', 'Universalism–nature', 'Universalism–tolerance']}
Loaded Track1 compact model from: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track1/encoder/deberta_v2_xxlarge_moco_scl/best_macro_f1_model


  0%|          | 0/65 [00:00<?, ?it/s]

,idx,Value,cand_id,candidate,pred_value,is_match,target_prob,pred_prob,margin,word_count,rerank_score
0,0,Conformity–interpersonal,0,I would agree to the frequent updates to ensur...,Conformity–interpersonal,True,0.999649,0.999649,0.999545,25,1.249536
1,0,Conformity–interpersonal,1,I would prioritize maintaining harmony by acco...,Conformity–interpersonal,True,0.999602,0.999602,0.999529,27,1.249485
2,0,Conformity–interpersonal,2,I would accommodate their need for frequent co...,Conformity–interpersonal,True,0.999621,0.999621,0.999507,27,1.249498
3,0,Conformity–interpersonal,3,I would agree to provide regular updates and a...,Conformity–interpersonal,True,0.999679,0.999679,0.999625,32,1.249585
4,1,Conformity–interpersonal,0,I would carefully choose my words to ensure I ...,Conformity–interpersonal,True,0.999637,0.999637,0.999574,32,1.249530


In [5]:
# Track1 compact-model sanity check. This replaces the old config-writing scratch cell.
best_metric_path = TRACK1_MODEL_DIR / "best_metric.json"
if best_metric_path.exists():
    with open(best_metric_path, encoding="utf-8") as f:
        print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
else:
    print(f"No best_metric.json found yet under {TRACK1_MODEL_DIR}")


{
  "best_macro_f1": 0.9417667224757209,
  "global_step": 770,
  "epoch": 14.0,
  "model_name": "microsoft/deberta-v2-xxlarge",
  "num_labels": 19,
  "labels": [
    "Self-direction–thought",
    "Self-direction–action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power–dominance",
    "Power–resources",
    "Face",
    "Security–personal",
    "Security–societal",
    "Tradition",
    "Conformity–rules",
    "Conformity–interpersonal",
    "Humility",
    "Benevolence–dependability",
    "Benevolence–caring",
    "Universalism–concern",
    "Universalism–nature",
    "Universalism–tolerance"
  ]
}


In [6]:
selected_df = (
    candidate_df
    .sort_values(["idx", "rerank_score"], ascending=[True, False])
    .groupby("idx", as_index=False)
    .first()
)

records = []
for row in dev_rows:
    selected = selected_df[selected_df["idx"] == row["idx"]].iloc[0]
    records.append({
        "idx": row["idx"],
        "Value": row["Value"],
        "pred_value": selected["pred_value"],
        "is_match": selected["is_match"],
        "track1_target_prob": selected["target_prob"],
        "track1_pred_prob": selected["pred_prob"],
        "margin": selected["margin"],
        "rerank_score": selected["rerank_score"],
        "word_count": selected["word_count"],
        "num_candidates": len(generated_rows[row["idx"]]["candidates"]),
        "Question": row["Question"],
        "reference_response": row["Consistent Value Response"],
        "generated_response": selected["candidate"],
    })

eval_df = pd.DataFrame(records)
eval_df.to_csv(DETAIL_FILE, index=False)

summary = pd.DataFrame([{
    "experiment": EXPERIMENT_NAME,
    "n": len(eval_df),
    "track1_match_rate": eval_df["is_match"].mean(),
    "avg_target_prob": eval_df["track1_target_prob"].mean(),
    "avg_margin": eval_df["margin"].mean(),
    "avg_word_count": eval_df["word_count"].mean(),
    "avg_num_candidates": eval_df["num_candidates"].mean(),
}]).round(4)

by_value = (
    eval_df
    .groupby("Value")
    .agg(
        n=("idx", "count"),
        match_rate=("is_match", "mean"),
        avg_target_prob=("track1_target_prob", "mean"),
        avg_margin=("margin", "mean"),
        avg_word_count=("word_count", "mean"),
    )
    .reset_index()
    .sort_values("match_rate")
    .round(4)
)

display(summary)
display(by_value)

summary.to_csv(EVAL_OUT_DIR / f"{EXPERIMENT_NAME}_summary.csv", index=False)
by_value.to_csv(EVAL_OUT_DIR / f"{EXPERIMENT_NAME}_by_value.csv", index=False)

print("Saved details:", DETAIL_FILE)

,experiment,n,track1_match_rate,avg_target_prob,avg_margin,avg_word_count,avg_num_candidates
0,grpo_sample4,514,1.0,0.9989,0.9983,33.0525,3.9961


,Value,n,match_rate,avg_target_prob,avg_margin,avg_word_count
0,Achievement,26,1.0,0.9997,0.9997,35.2308
1,Benevolence–caring,46,1.0,0.9994,0.9992,34.5870
2,Benevolence–dependability,27,1.0,0.9996,0.9994,36.0370
3,Conformity–interpersonal,34,1.0,0.9997,0.9995,32.7941
4,Conformity–rules,55,1.0,0.9996,0.9995,30.8364
5,Face,37,1.0,0.9997,0.9996,34.2703
6,Hedonism,24,1.0,0.9995,0.9994,32.9583
7,Humility,15,1.0,0.9996,0.9995,32.5333
8,Power–dominance,23,1.0,0.9995,0.9993,34.8696
9,Power–resources,35,1.0,0.9996,0.9995,35.7143


Saved details: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/eval_based_moco_scl/grpo/qwen35_9b_track1_eval_details.csv
